In [2]:
!pip install transformers peft trl datasets accelerate matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.0 MB/s  0:00:00


In [3]:
import torch, transformers, peft, trl, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("trl", trl.__version__)
print("datasets", datasets.__version__)
print("GPU available:", torch.cuda.is_available())

torch 2.14.1+cu130
transformers 5.22.1
peft 0.21.2
trl 1.18.2
datasets 4.8.5
GPU available: True


In [4]:
from datasets import load_dataset

ner = load_dataset("josephimperial/CebuaNER")
read = load_dataset("filbench/cebuano-readability")

print(ner)
print(read)

README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

cebuaner_final_annotated.txt:   0%|          | 0.00/2.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/188291 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  233kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/350 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 188291
    })
})
DatasetDict({
    test: Dataset({
        features: ['source', 'text', 'label'],
        num_rows: 350
    })
})


In [5]:
for i in range(25):
    print(i, repr(ner["train"][i]["text"]))

0 '-DOCSTART- -X- O'
1 "Aduna'y -X- _ O"
2 'pagsaka -X- _ O'
3 'sa -X- _ O'
4 'presyo -X- _ O'
5 'sa -X- _ O'
6 'lana -X- _ O'
7 ', -X- _ O'
8 'epektibo -X- _ O'
9 'sa -X- _ O'
10 'Martes -X- _ O'
11 ', -X- _ O'
12 'Hunyo -X- _ O'
13 '27 -X- _ O'
14 ', -X- _ O'
15 '2023 -X- _ O'
16 '. -X- _ O'
17 ''
18 'MOTORISTA -X- _ O'
19 ', -X- _ O'
20 'PATAY -X- _ O'
21 'HUMAN -X- _ O'
22 'MALIGSAN -X- _ O'
23 'ANG -X- _ O'
24 'ULO -X- _ O'


In [6]:
import re

sentences = []
current = []

for row in ner["train"]["text"]:
    row = row.strip()
    if row == "" or row.startswith("-DOCSTART-"):
        if current:
            sentences.append(" ".join(current))
            current = []
        continue
    current.append(row.split(" ")[0])

if current:
    sentences.append(" ".join(current))

sentences = [re.sub(r"\s+([,.;:!?%)\]])", r"\1", s) for s in sentences]

print("Total sentences:", len(sentences))
for s in sentences[:3]:
    print(s)
    print("---")

Total sentences: 4258
Aduna'y pagsaka sa presyo sa lana, epektibo sa Martes, Hunyo 27, 2023.
---
MOTORISTA, PATAY HUMAN MALIGSAN ANG ULO SA 10-WHEELER DUMP TRUCK Patay ang usa ka motorista sa usa ka vehicular accident ang nahitabo sa Barangay Caidiocan sa Valencia mga alas-6:20 sa gabii niadtong Sabado, Hunyo 24,2023. Giila ang biktima nga si Ruperto Solamillo Ausejo Jr., 46 anyos, ulitawo, ug lumolupyo sa Barangay Pulang Bato sa naasoy nga lungsod. Sumala pa sa imbestigasyon sa kapulisan sa Valencia, nagbiyahe ang duha sa managlahing direksyon. Pag-abot sa nahisgutang lugar, ni-overtake si Ausejo sa truck ug nawad-an og kontrol sa gimanehong motorsiklo nga niresulta sa iyang pagkatumba sa karsada. Tungod niini, aksidenteng naligsan sa 10-wheeler dump truck ang ulo sa biktima hinungdan sa iyang hinanaling kamatayon.
---
9 KA MAYOR SA NEGOR, NANAWAGAN NGA I-POSTPONE ANG BARANGAY, SK ELECTION Nipadayag og suporta ang siyam ka mga mayor sa Negros Oriental sa sugyot nga i-postpone ang elek

In [7]:
def tidy(s):
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r'"\s*([^"]*?)\s*"', r'"\1"', s)
    return s

articles = [tidy(s) for s in sentences]

print(articles[2][:400])

9 KA MAYOR SA NEGOR, NANAWAGAN NGA I-POSTPONE ANG BARANGAY, SK ELECTION Nipadayag og suporta ang siyam ka mga mayor sa Negros Oriental sa sugyot nga i-postpone ang eleksyon sa Barangay ug Sangguniang Kabataan (SK) sa probinsya nga gitakda karong Oktubre. Ilang gisubli ang mga kabalaka bahin sa seguridad lakip na ang "deep-rooted fear" sa mga residente sa probinsya human sa pagpatay kang Gov. Roel 


In [8]:
passages = [re.sub(r"\s+", " ", t).strip() for t in read["test"]["text"]]
passages = [p for p in passages if p]

docs = articles + passages

with open("cebuano_corpus.txt", "w", encoding="utf-8") as f:
    for d in docs:
        f.write(d + "\n")

print("News articles:", len(articles))
print("Readability passages:", len(passages))
print("Total documents:", len(docs))
print("Total words (approx):", sum(len(d.split()) for d in docs))

News articles: 4258
Readability passages: 350
Total documents: 4608
Total words (approx): 236661


In [9]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
os.makedirs("/content/drive/MyDrive/cebuano_project", exist_ok=True)
shutil.copy("cebuano_corpus.txt", "/content/drive/MyDrive/cebuano_project/cebuano_corpus.txt")
print("Saved to Drive")

Mounted at /content/drive
Saved to Drive


In [10]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("distilgpt2")

cebuano = ["kumusta", "salamat", "pagkaon", "mag-eskwela", "kabalaka", "gihatagan", "nagbiyahe", "lumolupyo"]
english = ["hello", "thanks", "eating", "schooling", "worry", "provided", "traveling", "resident"]

print("CEBUANO")
for w in cebuano:
    t = tok.tokenize(" " + w)
    print(f"{w:15} {len(t)} tokens  {t}")

print("\nENGLISH")
for w in english:
    t = tok.tokenize(" " + w)
    print(f"{w:15} {len(t)} tokens  {t}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

CEBUANO
kumusta         4 tokens  ['Ġk', 'um', 'ust', 'a']
salamat         3 tokens  ['Ġsal', 'am', 'at']
pagkaon         3 tokens  ['Ġpag', 'ka', 'on']
mag-eskwela     5 tokens  ['Ġmag', '-', 'es', 'kw', 'ela']
kabalaka        3 tokens  ['Ġk', 'abal', 'aka']
gihatagan       4 tokens  ['Ġg', 'ih', 'at', 'agan']
nagbiyahe       5 tokens  ['Ġn', 'ag', 'bi', 'ya', 'he']
lumolupyo       4 tokens  ['Ġlum', 'ol', 'up', 'yo']

ENGLISH
hello           1 tokens  ['Ġhello']
thanks          1 tokens  ['Ġthanks']
eating          1 tokens  ['Ġeating']
schooling       1 tokens  ['Ġschooling']
worry           1 tokens  ['Ġworry']
provided        1 tokens  ['Ġprovided']
traveling       1 tokens  ['Ġtraveling']
resident        1 tokens  ['Ġresident']


In [11]:
import random
random.seed(42)

sample_docs = random.sample(docs, 300)

ceb_words = sum(len(d.split()) for d in sample_docs)
ceb_tokens = sum(len(tok.encode(d)) for d in sample_docs)
print("Cebuano tokens per word:", round(ceb_tokens / ceb_words, 2))

english_text = """The price of oil went up on Tuesday, according to the government. A motorist died after a road accident in the village late on Saturday night. Police said the two men were traveling in different directions when the truck hit the motorcycle. Local mayors asked the province to postpone the election because of safety concerns."""

eng_words = len(english_text.split())
eng_tokens = len(tok.encode(english_text))
print("English tokens per word:", round(eng_tokens / eng_words, 2))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1552 > 1024). Running this sequence through the model will result in indexing errors


Cebuano tokens per word: 2.22
English tokens per word: 1.11


In [12]:
import torch
from transformers import AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
base_model.eval()

prompts = [
    "Kumusta ka?",
    "Ang mga tawo sa Cebu",
    "Sa pagsugod sa klase, ang mga estudyante",
    "Nipadayag ang mayor nga",
]

for p in prompts:
    inputs = tok(p, return_tensors="pt").to(device)
    out = base_model.generate(**inputs, max_new_tokens=40, do_sample=False,
                              pad_token_id=tok.eos_token_id)
    print("PROMPT:", p)
    print(tok.decode(out[0], skip_special_tokens=True))
    print("-" * 60)

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

PROMPT: Kumusta ka?
Kumusta ka?








































------------------------------------------------------------
PROMPT: Ang mga tawo sa Cebu
Ang mga tawo sa Cebu.







































------------------------------------------------------------
PROMPT: Sa pagsugod sa klase, ang mga estudyante
Sa pagsugod sa klase, ang mga estudyante, ang mga estudyante, ang mga estudyante, ang mga estudyante, ang mga estudyante, ang mga estudyante
------------------------------------------------------------
PROMPT: Nipadayag ang mayor nga
Nipadayag ang mayor nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga nga
------------------------------------------------------------


In [13]:
import json

baseline = {"tokens_per_word_cebuano": 2.22, "tokens_per_word_english": 1.11, "generations": {}}

for p in prompts:
    inputs = tok(p, return_tensors="pt").to(device)
    out = base_model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
    baseline["generations"][p] = tok.decode(out[0], skip_special_tokens=True)

with open("step0_baseline.json", "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)

shutil.copy("step0_baseline.json", "/content/drive/MyDrive/cebuano_project/step0_baseline.json")
print(baseline["generations"]["Kumusta ka?"])

Kumusta ka?










































In [14]:
from getpass import getpass
token = getpass("GitHub token: ")
!git clone https://{token}@github.com/CV0025Yuii/Teaching-an-LLM-with-Cebuano.git /content/repo
%cd /content/repo
!git config user.name "Shierwin"
!git config user.email "shierwinnino@gmail.com"
!git checkout shierwin || git checkout -b shierwin

GitHub token: ··········
Cloning into '/content/repo'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.
/content/repo
branch 'shierwin' set up to track 'origin/shierwin'.
Switched to a new branch 'shierwin'


In [15]:
import shutil, os
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
shutil.copy("/content/drive/MyDrive/cebuano_project/cebuano_corpus.txt", "data/cebuano_corpus.txt")
shutil.copy("/content/drive/MyDrive/cebuano_project/step0_baseline.json", "results/step0_baseline.json")

'results/step0_baseline.json'

In [16]:
shutil.copy("/content/Instructions.txt", "Instructions.txt")

'Instructions.txt'